# SmartCHDR — Model 3: Sri Lankan Meal Plan Recommender

## Purpose
Build a Content Based Filtering recommender that generates
personalised 7-day Sri Lankan meal plans for children aged 0-5.

## How It Works
1. Calculate child's daily nutritional needs (WHO/FAO guidelines)
2. Compare needs against meal dataset using cosine similarity
3. Select best matching meals per meal type
4. Generate 7-day meal plan

## Input
- age_months, gender, nutrition_risk (from Model 2)

## Output
- Personalised 7-day meal plan with Sri Lankan meals

## Dataset
- File: datasets/model3_meals/model3_meal_dataset.csv
- 70 Sri Lankan child-friendly meals with full nutritional values

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import json
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
# Load meal dataset
meals = pd.read_csv("../datasets/model3_meals/model3_meal_dataset.csv")

print(f"Total meals: {len(meals)}")
print(f"\nColumns: {meals.columns.tolist()}")
print(f"\nMeal types:")
print(meals['meal_type'].value_counts())
print(f"\nAge group coverage:")
print(f"  Min age: {meals['age_group_min'].min()} months")
print(f"  Max age: {meals['age_group_max'].max()} months")
print(f"\nFirst 3 meals:")
print(meals[['meal_id','meal_name','meal_type',
             'calories','protein_g','iron_mg',
             'calcium_mg','vitamin_a_ug']].head(3))

Total meals: 70

Columns: ['meal_id', 'meal_name', 'meal_type', 'age_group_min', 'age_group_max', 'calories', 'protein_g', 'carbohydrates_g', 'fat_g', 'fiber_g', 'iron_mg', 'calcium_mg', 'vitamin_a_ug', 'vitamin_c_mg', 'zinc_mg', 'is_vegetarian', 'is_vegan', 'is_gluten_free', 'primary_ingredient', 'description']

Meal types:
meal_type
lunch        23
breakfast    21
snack        14
dinner       12
Name: count, dtype: int64

Age group coverage:
  Min age: 4 months
  Max age: 60 months

First 3 meals:
   meal_id                           meal_name  meal_type  calories  \
0        1       Rice Porridge (Haal Paayasam)  breakfast       120   
1        2  Kola Kanda (Green Herbal Porridge)  breakfast        95   
2        3            Oat Porridge with Banana  breakfast       145   

   protein_g  iron_mg  calcium_mg  vitamin_a_ug  
0        2.1      0.8          20             5  
1        2.8      2.1          85           320  
2        3.5      1.4          35             8  


In [3]:
def get_daily_nutritional_needs(age_months, gender, nutrition_risk):
    """
    Returns daily nutritional needs for a child based on
    WHO/FAO dietary reference values.
    
    Nutrition risk adjusts targets upward:
    - low: standard requirements
    - moderate: +15% on key nutrients
    - high: +25% on key nutrients
    """
    
    # Base nutritional needs by age group (per day)
    # Source: WHO/FAO Human Vitamin and Mineral Requirements
    if age_months < 6:
        needs = {
            'calories'        : 550,
            'protein_g'       : 9.1,
            'carbohydrates_g' : 60.0,
            'fat_g'           : 31.0,
            'fiber_g'         : 0.0,
            'iron_mg'         : 0.27,
            'calcium_mg'      : 200.0,
            'vitamin_a_ug'    : 400.0,
            'vitamin_c_mg'    : 40.0,
            'zinc_mg'         : 2.0
        }
    elif age_months < 12:
        needs = {
            'calories'        : 700,
            'protein_g'       : 11.0,
            'carbohydrates_g' : 95.0,
            'fat_g'           : 30.0,
            'fiber_g'         : 2.5,
            'iron_mg'         : 11.0,
            'calcium_mg'      : 260.0,
            'vitamin_a_ug'    : 500.0,
            'vitamin_c_mg'    : 50.0,
            'zinc_mg'         : 3.0
        }
    elif age_months < 24:
        needs = {
            'calories'        : 1000,
            'protein_g'       : 13.0,
            'carbohydrates_g' : 130.0,
            'fat_g'           : 30.0,
            'fiber_g'         : 19.0,
            'iron_mg'         : 7.0,
            'calcium_mg'      : 700.0,
            'vitamin_a_ug'    : 300.0,
            'vitamin_c_mg'    : 15.0,
            'zinc_mg'         : 3.0
        }
    elif age_months < 36:
        needs = {
            'calories'        : 1200,
            'protein_g'       : 13.0,
            'carbohydrates_g' : 160.0,
            'fat_g'           : 35.0,
            'fiber_g'         : 19.0,
            'iron_mg'         : 7.0,
            'calcium_mg'      : 700.0,
            'vitamin_a_ug'    : 300.0,
            'vitamin_c_mg'    : 15.0,
            'zinc_mg'         : 3.0
        }
    elif age_months < 48:
        needs = {
            'calories'        : 1400,
            'protein_g'       : 19.0,
            'carbohydrates_g' : 190.0,
            'fat_g'           : 39.0,
            'fiber_g'         : 25.0,
            'iron_mg'         : 10.0,
            'calcium_mg'      : 1000.0,
            'vitamin_a_ug'    : 400.0,
            'vitamin_c_mg'    : 25.0,
            'zinc_mg'         : 5.0
        }
    else:
        needs = {
            'calories'        : 1600,
            'protein_g'       : 19.0,
            'carbohydrates_g' : 220.0,
            'fat_g'           : 44.0,
            'fiber_g'         : 25.0,
            'iron_mg'         : 10.0,
            'calcium_mg'      : 1000.0,
            'vitamin_a_ug'    : 400.0,
            'vitamin_c_mg'    : 25.0,
            'zinc_mg'         : 5.0
        }
    
    # Apply nutrition risk multiplier
    risk_multiplier = {'low': 1.0, 'moderate': 1.15, 'high': 1.25}
    multiplier = risk_multiplier.get(nutrition_risk, 1.0)
    
    # Apply multiplier to key nutrients only (not calories)
    for nutrient in ['protein_g','iron_mg','calcium_mg',
                     'vitamin_a_ug','vitamin_c_mg','zinc_mg']:
        needs[nutrient] = round(needs[nutrient] * multiplier, 2)
    
    return needs

# Test the function
test_needs = get_daily_nutritional_needs(12, 'male', 'high')
print("Daily needs for 12-month boy with HIGH nutrition risk:")
for nutrient, value in test_needs.items():
    print(f"  {nutrient:20s}: {value}")

Daily needs for 12-month boy with HIGH nutrition risk:
  calories            : 1000
  protein_g           : 16.25
  carbohydrates_g     : 130.0
  fat_g               : 30.0
  fiber_g             : 19.0
  iron_mg             : 8.75
  calcium_mg          : 875.0
  vitamin_a_ug        : 375.0
  vitamin_c_mg        : 18.75
  zinc_mg             : 3.75


In [4]:
# Nutritional columns used for similarity matching
NUTRITION_COLS = ['calories', 'protein_g', 'carbohydrates_g', 'fat_g',
                  'fiber_g', 'iron_mg', 'calcium_mg', 
                  'vitamin_a_ug', 'vitamin_c_mg', 'zinc_mg']

def get_eligible_meals(age_months, meal_type):
    """Filter meals suitable for this age and meal type."""
    eligible = meals[
        (meals['meal_type'] == meal_type) &
        (meals['age_group_min'] <= age_months) &
        (meals['age_group_max'] >= age_months)
    ].copy()
    return eligible

def recommend_meals(age_months, gender, nutrition_risk, 
                    meal_type, top_n=5):
    """
    Recommend top N meals for a given meal type using cosine similarity.
    
    Steps:
    1. Get daily nutritional needs for this child
    2. Calculate per-meal-type needs (breakfast=25%, lunch=35%, 
       dinner=25%, snack=15% of daily needs)
    3. Filter eligible meals by age and meal type
    4. Scale nutritional values using MinMaxScaler
    5. Calculate cosine similarity between child needs and each meal
    6. Return top N matches
    """
    
    # Daily needs for this child
    daily_needs = get_daily_nutritional_needs(
        age_months, gender, nutrition_risk)
    
    # Per meal type portion of daily needs
    meal_portions = {
        'breakfast': 0.25,
        'lunch'    : 0.35,
        'dinner'   : 0.25,
        'snack'    : 0.15
    }
    portion = meal_portions.get(meal_type, 0.25)
    
    # Calculate target nutrition for this meal
    target = {col: daily_needs.get(col, 0) * portion 
              for col in NUTRITION_COLS}
    target_vector = np.array([target[col] for col in NUTRITION_COLS])
    
    # Get eligible meals for this age and meal type
    eligible = get_eligible_meals(age_months, meal_type)
    
    if len(eligible) == 0:
        return pd.DataFrame()
    
    # Get nutritional matrix for eligible meals
    meal_matrix = eligible[NUTRITION_COLS].values
    
    # Stack target with meals for scaling together
    combined = np.vstack([target_vector, meal_matrix])
    
    # Scale to 0-1 range (so no single nutrient dominates)
    scaler = MinMaxScaler()
    combined_scaled = scaler.fit_transform(combined)
    
    target_scaled = combined_scaled[0].reshape(1, -1)
    meals_scaled  = combined_scaled[1:]
    
    # Calculate cosine similarity
    similarities = cosine_similarity(target_scaled, meals_scaled)[0]
    
    # Add similarity scores to eligible meals
    eligible = eligible.copy()
    eligible['similarity_score'] = similarities
    
    # Return top N meals sorted by similarity
    top_meals = eligible.nlargest(top_n, 'similarity_score')
    
    return top_meals[['meal_id', 'meal_name', 'meal_type',
                       'calories', 'protein_g', 'iron_mg',
                       'calcium_mg', 'vitamin_a_ug',
                       'similarity_score', 'description']]

# Test the recommender
print("Top 3 breakfast recommendations for 12-month boy, HIGH risk:")
results = recommend_meals(12, 'male', 'high', 'breakfast', top_n=3)
print(results.to_string(index=False))

print("\nTop 3 lunch recommendations for 24-month girl, LOW risk:")
results2 = recommend_meals(24, 'female', 'low', 'lunch', top_n=3)
print(results2.to_string(index=False))

Top 3 breakfast recommendations for 12-month boy, HIGH risk:
 meal_id                      meal_name meal_type  calories  protein_g  iron_mg  calcium_mg  vitamin_a_ug  similarity_score                                                                description
       6       Ragi (Kurakkan) Porridge breakfast       155        3.8      3.0         200             0          0.876187 Finger millet porridge, exceptionally high in calcium for bone development
       3       Oat Porridge with Banana breakfast       145        3.5      1.4          35             8          0.847974         Smooth oat porridge blended with ripe banana for natural sweetness
      64 Coconut Flour Roti with Banana breakfast       215        3.8      1.2          18             5          0.825433                                 Gluten free coconut flour roti with banana

Top 3 lunch recommendations for 24-month girl, LOW risk:
 meal_id                               meal_name meal_type  calories  protein_g  iron

In [8]:
def generate_7day_meal_plan(age_months, gender, nutrition_risk):
    """
    Generate a personalised 7-day Sri Lankan meal plan.
    - Gets top 14 recommendations per meal type
    - Blocks meals used in last 2 days to avoid repetition
    - Ensures maximum variety across the week
    """
    
    np.random.seed(42)
    
    meal_types = ['breakfast', 'lunch', 'dinner', 'snack']
    
    # Get more recommendations for better variety
    recommendations = {}
    for meal_type in meal_types:
        recs = recommend_meals(
            age_months, gender, nutrition_risk,
            meal_type, top_n=14)
        recommendations[meal_type] = recs
    
    # Build 7-day plan
    meal_plan = []
    used_meals = {mt: [] for mt in meal_types}
    
    days = ['Monday','Tuesday','Wednesday',
            'Thursday','Friday','Saturday','Sunday']
    
    for day in days:
        day_plan = {'day': day}
        
        for meal_type in meal_types:
            recs = recommendations[meal_type]
            
            if len(recs) == 0:
                day_plan[meal_type] = 'Not available'
                continue
            
            # Block meals used in last 2 days (not just last 1)
            recent = used_meals[meal_type][-2:]
            available = recs[~recs['meal_name'].isin(recent)]
            
            # If not enough available, relax to last 1 day
            if len(available) == 0:
                recent = used_meals[meal_type][-1:]
                available = recs[~recs['meal_name'].isin(recent)]
            
            # If still empty, use all recommendations
            if len(available) == 0:
                available = recs
            
            # Weighted random selection from available meals
            weights = np.array(available['similarity_score'].values,
                               dtype=float)
            weights = weights / weights.sum()
            
            chosen_idx = np.random.choice(len(available), p=weights)
            chosen = available.iloc[chosen_idx]
            
            day_plan[meal_type]          = chosen['meal_name']
            day_plan[f'{meal_type}_cal'] = chosen['calories']
            
            used_meals[meal_type].append(chosen['meal_name'])
        
        # Calculate daily totals
        day_plan['total_calories'] = (
            day_plan.get('breakfast_cal', 0) +
            day_plan.get('lunch_cal',     0) +
            day_plan.get('dinner_cal',    0) +
            day_plan.get('snack_cal',     0)
        )
        
        meal_plan.append(day_plan)
    
    return pd.DataFrame(meal_plan)

# Test with a 12-month boy, high nutrition risk
print("=" * 70)
print("7-DAY MEAL PLAN — 12 month boy, HIGH nutrition risk")
print("=" * 70)
plan = generate_7day_meal_plan(12, 'male', 'high')

for _, row in plan.iterrows():
    print(f"\n{row['day']}")
    print(f"  Breakfast : {row['breakfast']}")
    print(f"  Lunch     : {row['lunch']}")
    print(f"  Dinner    : {row['dinner']}")
    print(f"  Snack     : {row['snack']}")
    print(f"  Total Cal : {row['total_calories']} kcal")

7-DAY MEAL PLAN — 12 month boy, HIGH nutrition risk

Monday
  Breakfast : Dhal (Parippu) with Rice Roti
  Lunch     : Rice with Beetroot Curry
  Dinner    : Chicken Soup with Vegetables
  Snack     : Boiled Jak Fruit (Kos)
  Total Cal : 880 kcal

Tuesday
  Breakfast : Oat Porridge with Banana
  Lunch     : Rice with Mixed Vegetable Curry and Egg
  Dinner    : Rice with Tempered Dhal and Coconut Sambol
  Snack     : Gotukola Salad (Gotu Kola Sambol)
  Total Cal : 950 kcal

Wednesday
  Breakfast : Pittu with Coconut Milk and Banana
  Lunch     : Rice with Cowpea (Barbati) Curry
  Dinner    : String Hoppers with Dhal Curry
  Snack     : Mango (Amba)
  Total Cal : 940 kcal

Thursday
  Breakfast : String Hoppers (Indiappa) with Coconut Milk
  Lunch     : Egg Fried Rice
  Dinner    : Soft Rice with Fish and Vegetables
  Snack     : Coconut Milk Pudding (Watalappan Style)
  Total Cal : 1115 kcal

Friday
  Breakfast : Coconut Roti (Pol Roti)
  Lunch     : Rice with Dhal Curry
  Dinner    : Ric

In [6]:
test_profiles = [
    {'age_months': 8,  'gender': 'female', 
     'nutrition_risk': 'high',
     'label': '8-month girl, HIGH risk'},
    {'age_months': 24, 'gender': 'male',   
     'nutrition_risk': 'moderate',
     'label': '24-month boy, MODERATE risk'},
    {'age_months': 48, 'gender': 'female', 
     'nutrition_risk': 'low',
     'label': '48-month girl, LOW risk'},
]

for profile in test_profiles:
    print("=" * 70)
    print(f"MEAL PLAN — {profile['label']}")
    print("=" * 70)
    plan = generate_7day_meal_plan(
        profile['age_months'],
        profile['gender'],
        profile['nutrition_risk']
    )
    # Show first 3 days only for brevity
    for _, row in plan.head(3).iterrows():
        print(f"\n  {row['day']}")
        print(f"    Breakfast : {row['breakfast']}")
        print(f"    Lunch     : {row['lunch']}")
        print(f"    Dinner    : {row['dinner']}")
        print(f"    Snack     : {row['snack']}")
        print(f"    Total Cal : {row['total_calories']} kcal")
    print()

MEAL PLAN — 8-month girl, HIGH risk

  Monday
    Breakfast : Kiribath (Milk Rice)
    Lunch     : Mashed Fish with Rice
    Dinner    : Dhal Soup (Parippu Soup)
    Snack     : Ripe Papaya and Curd
    Total Cal : 665 kcal

  Tuesday
    Breakfast : Kola Kanda (Green Herbal Porridge)
    Lunch     : Rice with Spinach (Nivithi) Curry
    Dinner    : Mashed Potato with Butter and Milk
    Snack     : Ripe Banana
    Total Cal : 625 kcal

  Wednesday
    Breakfast : Rice Porridge (Haal Paayasam)
    Lunch     : Mashed Egg Yolk with Rice
    Dinner    : Soft Rice with Fish and Vegetables
    Snack     : Papaya (Papol)
    Total Cal : 615 kcal

MEAL PLAN — 24-month boy, MODERATE risk

  Monday
    Breakfast : Coconut Milk Semolina (Rawa) Porridge
    Lunch     : Rice with Cowpea (Barbati) Curry
    Dinner    : Dhal Soup (Parippu Soup)
    Snack     : Boiled Jak Fruit (Kos)
    Total Cal : 790 kcal

  Tuesday
    Breakfast : Coconut Flour Roti with Banana
    Lunch     : Rice with Mixed Veg

In [7]:
os.makedirs('../ml_models', exist_ok=True)

# Save the meals dataframe (Flask will load this)
meals.to_csv('../ml_models/model3_meal_dataset.pkl', index=False)

# Save nutrition columns list
with open('../ml_models/model3_nutrition_cols.pkl', 'wb') as f:
    pickle.dump(NUTRITION_COLS, f)

# Save the scaler fitted on full meal dataset
# This ensures consistent scaling in Flask
full_meal_matrix = meals[NUTRITION_COLS].values
scaler_full = MinMaxScaler()
scaler_full.fit(full_meal_matrix)
with open('../ml_models/model3_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_full, f)

print("✅ Model 3 files saved to ml_models/:")
print("  - model3_meal_dataset.pkl  (meal data)")
print("  - model3_nutrition_cols.pkl (feature columns)")
print("  - model3_scaler.pkl         (fitted scaler)")

# Verify files
for fname in ['model3_meal_dataset.pkl',
              'model3_nutrition_cols.pkl',
              'model3_scaler.pkl']:
    path = f'../ml_models/{fname}'
    size = os.path.getsize(path)
    print(f"  {fname}: {size/1024:.1f} KB")

✅ Model 3 files saved to ml_models/:
  - model3_meal_dataset.pkl  (meal data)
  - model3_nutrition_cols.pkl (feature columns)
  - model3_scaler.pkl         (fitted scaler)
  model3_meal_dataset.pkl: 11.2 KB
  model3_nutrition_cols.pkl: 0.1 KB
  model3_scaler.pkl: 0.9 KB


## Model 3 Complete ✅

### How It Works
1. Child's age, gender, nutrition_risk → daily nutritional needs
2. Per-meal-type targets calculated (breakfast=25%, lunch=35%, 
   dinner=25%, snack=15%)
3. Eligible meals filtered by age suitability
4. Cosine similarity computed between child needs and each meal
5. Top meals selected with variety across 7 days

### Files Saved
| File | Purpose |
|---|---|
| model3_meal_dataset.pkl | Full meal dataset for Flask |
| model3_nutrition_cols.pkl | Nutritional feature columns |
| model3_scaler.pkl | MinMaxScaler for consistent scaling |

### How Flask Will Use This
1. Load meal dataset and scaler at startup
2. Accept age, gender, nutrition_risk from frontend
3. Calculate nutritional needs
4. Run cosine similarity
5. Return 7-day meal plan as JSON